# 🦷 Dental AI - YOLOv8x Training on DENTEX Dataset

**Kaggle GPU Training** - 30h GPU/week free (P100/T4)

---

## 📋 Prerequisites

- Kaggle API token (Settings → API → Create New Token)
- Dataset: DENTEX Challenge 2023 (705 train + 50 val + 250 test)
- Model: YOLOv8x (extra-large) or YOLOv8m for faster training

## 🚀 Quick Start

1. **Fork this notebook** to your Kaggle account
2. **Add dataset**: `truthisneverlinear/dentex-challenge-2023`
3. **Enable GPU**: Settings → Accelerator → GPU P100/T4
4. **Run all cells**

## 📊 Expected Results

| Metric | Target |
|--------|--------|
| mAP50 | ≥ 0.50 |
| Recall (Caries) | ≥ 0.65 |
| Training time | ~2-4 hours (100 epochs) |


In [ ]:
# ============================================================
# 🔧 SETUP & INSTALLATION
# ============================================================

!pip install -q ultralytics==8.4.114 opencv-python-headless pyyaml tqdm

import os
import sys
import json
import shutil
from pathlib import Path

print("✅ Dependencies installed")

In [ ]:
# ============================================================
# 📊 CHECK DISK SPACE
# ============================================================

import shutil

def get_disk_usage(path):
    total, used, free = shutil.disk_usage(path)
    return total // (2**30), used // (2**30), free // (2**30)

total, used, free = get_disk_usage("/kaggle/working")
print(f"📊 Disk: {free} GB free / {total} GB total")

if free < 10:
    print("⚠️ WARNING: Less than 10GB free! Consider deleting old outputs.")

In [ ]:
# ============================================================
# 🔐 KAGGLE AUTHENTICATION
# ============================================================

import os
from kaggle.api.kaggle_api_extended import KaggleApi

# Set your Kaggle credentials
os.environ["KAGGLE_USERNAME"] = "eriksmite"
os.environ["KAGGLE_KEY"] = "YOUR_KAGGLE_KEY_HERE"  # ⚠️ ZADAJ SVOJ KLÚČ

# Verify authentication
api = KaggleApi()
api.authenticate()
print(f"✅ Authenticated as: {api.config_values.get('username')}")

In [ ]:
# ============================================================
# 📥 DOWNLOAD & EXTRACT DENTEX (MEMORY-EFFICIENT)
# ============================================================

import zipfile
import json
from pathlib import Path

WORK_DIR = "/kaggle/working"
DATASET_URL = 'truthisneverlinear/dentex-challenge-2023'

# Step 1: Download only the training data (not validation/test)
print("📥 Downloading DENTEX training data...")
api.dataset_download_files(
    DATASET_URL,
    path=WORK_DIR,
    force=True,
    quiet=False
)

# Step 2: Extract only what we need (skip validation/test)
zip_path = Path(WORK_DIR) / "dentex-challenge-2023.zip"
if zip_path.exists():
    print("📦 Extracting training data only...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # List contents to find training data
        all_files = zip_ref.namelist()
        
        # Extract only training images and annotations
        train_files = [f for f in all_files if 'training_data' in f]
        print(f"  Found {len(train_files)} training files")
        
        # Extract training data
        zip_ref.extractall(WORK_DIR, train_files)
        print("  ✅ Training data extracted")
    
    # Delete zip to save space
    os.remove(zip_path)
    print("  🗑️ Deleted zip file")

# Step 3: Find the extracted files
DATASET_PATH = None
for possible_path in [
    Path(WORK_DIR) / "training_data",
    Path(WORK_DIR) / "dentex-challenge-2023" / "training_data",
]:
    if possible_path.exists():
        DATASET_PATH = possible_path
        break

if DATASET_PATH is None:
    # Search for the JSON file
    for root, dirs, files in os.walk(WORK_DIR):
        for f in files:
            if f.endswith('.json') and 'quadrant' in f.lower():
                DATASET_PATH = Path(root).parent
                break

print(f"\n📂 Dataset path: {DATASET_PATH}")

# Show structure
if DATASET_PATH:
    for root, dirs, files in os.walk(DATASET_PATH):
        level = root.replace(str(DATASET_PATH), '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        if level < 2:
            for f in files[:5]:
                print(f"{indent}  {f}")

In [ ]:
# ============================================================
# 🔄 CONVERT DENTEX TO YOLO FORMAT (MEMORY-EFFICIENT)
# ============================================================

import json
import shutil
from pathlib import Path
import os

# Find the source directories
SRC_DIR = None
for possible in [
    DATASET_PATH / "training_data" / "training_data" / "quadrant-enumeration-disease",
    DATASET_PATH / "quadrant-enumeration-disease",
    DATASET_PATH,
]:
    if (possible / "train_quadrant_enumeration_disease.json").exists():
        SRC_DIR = possible
        break

if SRC_DIR is None:
    # Search for the JSON
    for root, dirs, files in os.walk(str(DATASET_PATH)):
        if "train_quadrant_enumeration_disease.json" in files:
            SRC_DIR = Path(root)
            break

print(f"📂 Source dir: {SRC_DIR}")

YOLO_DIR = Path(WORK_DIR) / "dentex_yolo"
for split in ["train", "val"]:
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

def convert_coco_to_yolo(coco_ann, img_width, img_height):
    x, y, w, h = coco_ann['bbox']
    x_center = (x + w / 2) / img_width
    y_center = (y + h / 2) / img_height
    w_norm = w / img_width
    h_norm = h / img_height
    return x_center, y_center, w_norm, h_norm

# Process training data
train_json = SRC_DIR / "train_quadrant_enumeration_disease.json"
train_img_src = SRC_DIR / "xrays"

print(f"\n🔄 Converting training data...")
with open(train_json) as f:
    data = json.load(f)

annotations = {}
for ann in data['annotations']:
    img_id = ann['image_id']
    if img_id not in annotations:
        annotations[img_id] = []
    annotations[img_id].append(ann)

img_info = {img['id']: img for img in data['images']}

count = 0
for img_id, ann_list in annotations.items():
    if img_id not in img_info:
        continue
    
    info = img_info[img_id]
    src_img = train_img_src / info['file_name']
    if not src_img.exists():
        continue
    
    # Copy image to YOLO structure
    dst_img = YOLO_DIR / "images" / "train" / info['file_name']
    shutil.copy2(src_img, dst_img)
    
    # Create YOLO label file
    label_path = YOLO_DIR / "labels" / "train" / (Path(info['file_name']).stem + ".txt")
    with open(label_path, 'w') as f:
        for ann in ann_list:
            class_id = ann['category_id_3']  # 0=Impacted, 1=Caries, 2=Periapical, 3=Deep Caries
            xc, yc, w, h = convert_coco_to_yolo(ann, info['width'], info['height'])
            f.write(f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
    
    count += 1
    if count % 100 == 0:
        print(f"  Processed {count} images...")

print(f"\n✅ Training conversion complete: {count} images")

# Delete original training data to save space
print("\n🗑️ Deleting original training data...")
shutil.rmtree(train_img_src, ignore_errors=True)
if train_json.exists():
    os.remove(train_json)

# Check disk after conversion
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"📊 Disk after conversion: {free} GB free")

In [ ]:
# ============================================================
# 📝 CREATE DATA.YAML
# ============================================================

data_yaml = f"""
# DENTEX Dataset - YOLO Configuration
# Classes: 4 diagnoses from DENTEX categories_3
# 0: Impacted
# 1: Caries
# 2: Periapical Lesion
# 3: Deep Caries

path: {YOLO_DIR}  # dataset root
train: images/train  # train images
val: images/val      # val images

names:
  0: Impacted
  1: Caries
  2: Periapical Lesion
  3: Deep Caries

nc: 4
"""

yaml_path = YOLO_DIR / "data.yaml"
with open(yaml_path, 'w') as f:
    f.write(data_yaml)

print(f"✅ data.yaml created at: {yaml_path}")
print(data_yaml)

In [ ]:
# ============================================================
# 🏋️ TRAINING CONFIGURATION
# ============================================================

TRAIN_CONFIG = {
    "model": "yolov8x.pt",           # yolov8x.pt, yolov8m.pt, yolov8s.pt
    "data": str(YOLO_DIR / "data.yaml"),
    "epochs": 100,
    "imgsz": 1280,
    "batch": 16,                       # Reduce to 8 if OOM
    "device": 0,                       # GPU 0
    "project": "/kaggle/working/runs/detect",
    "name": "dentex_v1",
    "patience": 20,
    "lr0": 0.01,
    "cos_lr": True,
    "close_mosaic": 10,
    "workers": 4,
    "amp": True,
    "save": True,
    "save_period": 10,
    "cache": False,                    # Set True if enough RAM
    "verbose": True,
}

print("Training config:")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# ============================================================
# 🏋️ START TRAINING
# ============================================================

from ultralytics import YOLO

print("🚀 Starting training...")
print(f"Model: {TRAIN_CONFIG['model']}")
print(f"Epochs: {TRAIN_CONFIG['epochs']}")
print(f"Image size: {TRAIN_CONFIG['imgsz']}")
print(f"Batch size: {TRAIN_CONFIG['batch']}")

model = YOLO(TRAIN_CONFIG["model"])

results = model.train(
    data=TRAIN_CONFIG["data"],
    epochs=TRAIN_CONFIG["epochs"],
    imgsz=TRAIN_CONFIG["imgsz"],
    batch=TRAIN_CONFIG["batch"],
    device=TRAIN_CONFIG["device"],
    project=TRAIN_CONFIG["project"],
    name=TRAIN_CONFIG["name"],
    patience=TRAIN_CONFIG["patience"],
    lr0=TRAIN_CONFIG["lr0"],
    cos_lr=TRAIN_CONFIG["cos_lr"],
    close_mosaic=TRAIN_CONFIG["close_mosaic"],
    workers=TRAIN_CONFIG["workers"],
    amp=TRAIN_CONFIG["amp"],
    save=TRAIN_CONFIG["save"],
    save_period=TRAIN_CONFIG["save_period"],
    cache=TRAIN_CONFIG["cache"],
    verbose=TRAIN_CONFIG["verbose"],
)

print("✅ Training complete!")

In [ ]:
# ============================================================
# 📊 VALIDATION & METRICS
# ============================================================

print("📊 Running validation...")
metrics = model.val(data=TRAIN_CONFIG["data"], split="val", imgsz=TRAIN_CONFIG["imgsz"], batch=TRAIN_CONFIG["batch"])

print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

print("\nPer-class AP50:")
class_names = ["Impacted", "Caries", "Periapical Lesion", "Deep Caries"]
for i, (ap50, rec) in enumerate(zip(metrics.box.ap50, metrics.box.recall)):
    print(f"  {class_names[i]}: AP50={ap50:.4f}, Recall={rec:.4f}")

# Check targets
print("\n🎯 TARGET CHECK:")
print(f"  mAP50 ≥ 0.50: {'✅ PASS' if metrics.box.map50 >= 0.50 else '❌ FAIL'} ({metrics.box.map50:.4f})")
print(f"  Recall Caries ≥ 0.65: {'✅ PASS' if metrics.box.recall[1] >= 0.65 else '❌ FAIL'} ({metrics.box.recall[1]:.4f})")

In [ ]:
# ============================================================
# 💾 SAVE & UPLOAD BEST MODEL
# ============================================================

import shutil

BEST_MODEL = "/kaggle/working/runs/detect/dentex_v1/weights/best.pt"
LAST_MODEL = "/kaggle/working/runs/detect/dentex_v1/weights/last.pt"

if os.path.exists(BEST_MODEL):
    print(f"✅ Best model found: {BEST_MODEL}")
    print(f"   Size: {os.path.getsize(BEST_MODEL) / 1024 / 1024:.1f} MB")
else:
    print(f"❌ Best model not found at: {BEST_MODEL}")

# Upload to HuggingFace (optional)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    print("📤 Uploading to HuggingFace...")
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.upload_file(
            path_or_fileobj=BEST_MODEL,
            path_in_repo="yolov8x_dental.pt",
            repo_id="ericecek/dental-ai-models",
            repo_type="model",
        )
        print("✅ Uploaded to HuggingFace")
    except Exception as e:
        print(f"❌ Upload failed: {e}")
else:
    print("ℹ️ HF_TOKEN not set - skipping HuggingFace upload")

# Copy to Kaggle output for download
OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
shutil.copy2(BEST_MODEL, OUTPUT_DIR / "best.pt")
shutil.copy2(LAST_MODEL, OUTPUT_DIR / "last.pt")
print(f"✅ Models copied to {OUTPUT_DIR} for download")

In [ ]:
# ============================================================
# 📤 EXPORT FOR DEPLOYMENT
# ============================================================

# Export to ONNX for deployment
print("📦 Exporting to ONNX...")
onnx_path = model.export(format="onnx", imgsz=1280, simplify=True)
print(f"✅ ONNX model: {onnx_path}")

# Copy to output
onnx_file = Path(onnx_path)
if onnx_file.exists():
    shutil.copy2(onnx_file, OUTPUT_DIR / "best.onnx")
    print(f"✅ ONNX copied to {OUTPUT_DIR}/best.onnx")

# Print summary
print("\n" + "="*60)
print("🎉 TRAINING COMPLETE!")
print("="*60)
print(f"Best model: {BEST_MODEL}")
print(f"ONNX model: {onnx_path}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Config: {YOLO_DIR}/data.yaml")
print("\n📥 Download from Kaggle: Output tab → dentex_v1")
print("🔗 Or copy to backend/weights/yolov8x_dental.pt for deployment")